# EDA — Entendimento do problema e decisões para modelagem

## Predição e Inteligência Analítica para Alfabetização no Brasil

Este notebook realiza a **Análise Exploratória de Dados (EDA)** da base analítica enriquecida, composta por **1.966.605 alunos e 41 variáveis**.

A exploração foi desenhada como ponte entre a construção dos dados e a modelagem supervisionada. Portanto, não se limita à produção de gráficos: cada etapa gera evidências para decisões de pré-processamento, seleção de features, prevenção de *data leakage*, validação e interpretação do modelo.

### Pergunta analítica central

> Quais características educacionais, territoriais, socioeconômicas e financeiras estão associadas à alfabetização e podem apoiar uma predição válida, interpretável e útil para políticas públicas?

### Princípios metodológicos

- a unidade principal é o aluno;
- várias features enriquecidas possuem granularidade municipal e se repetem entre alunos;
- contagens e estatísticas principais são calculadas sobre a base completa;
- amostras reprodutíveis são usadas apenas em visualizações densas;
- valores ausentes e outliers são diagnosticados, não imputados nem removidos nesta etapa;
- associação não será interpretada como causalidade;
- a base enriquecida original será tratada como artefato imutável;
- todas as decisões para modelagem serão registradas explicitamente.

## 1. Configuração, reprodutibilidade e caminhos

Esta etapa centraliza parâmetros, estilos e caminhos utilizados na EDA.

O arquivo de entrada é o artefato final produzido pelo notebook de padronização. As saídas da exploração serão gravadas em uma pasta própria, sem sobrescrever a base enriquecida.

In [0]:
# Objetivo:
#
# Configurar o ambiente da EDA e definir caminhos,
# parâmetros e padrões visuais reprodutíveis.
#
# Justificativa:
#
# A centralização evita caminhos dispersos, garante
# amostragem determinística e mantém a base original
# protegida contra sobrescrita.
#
# Ação:
#
# Importa as bibliotecas, fixa a semente aleatória,
# define os caminhos oficiais e prepara a pasta de
# evidências da análise exploratória.

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda valor: f"{valor:,.4f}")

RANDOM_STATE = 42
TAMANHO_AMOSTRA_GRAFICOS = 100_000
MIN_ALUNOS_MUNICIPIO = 100

BASE_PATH = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase2"
)
GOLD_PATH = f"{BASE_PATH}/gold"

INPUT_PATH = (
    f"{GOLD_PATH}/alunos_base_enriquecida_finalizada/"
    "base_analitica_final.csv"
)
EDA_OUTPUT_PATH = f"{GOLD_PATH}/eda"

try:
    dbutils.fs.mkdirs(EDA_OUTPUT_PATH)
except NameError:
    Path(EDA_OUTPUT_PATH).mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
CORES_TARGET = ["#C44E52", "#4C72B0"]

print("Entrada:", INPUT_PATH)
print("Saídas da EDA:", EDA_OUTPUT_PATH)
print("Semente aleatória:", RANDOM_STATE)

## 2. Carregamento e validação estrutural

A validação inicial verifica o contrato estabelecido na construção da base:

- 1.966.605 registros;
- 41 variáveis;
- nomes únicos;
- target `in_alfabetizado`;
- blocos de enriquecimento identificáveis pelos prefixos `atlas_`, `censo_`, `fundeb_` e `indicador_`.

Esta etapa interrompe a execução caso um requisito crítico não seja atendido, evitando que a EDA produza resultados sobre uma base incorreta.

In [0]:
# Objetivo:
#
# Carregar a base final sem alterar o artefato
# produzido na etapa de enriquecimento.
#
# Justificativa:
#
# Toda a EDA depende da integridade da população,
# do schema e da variável-alvo.
#
# Ação:
#
# Lê o CSV consolidado e exibe sua dimensão.

df = pd.read_csv(
    INPUT_PATH,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Registros: {df.shape[0]:,}")
print(f"Variáveis: {df.shape[1]}")
df.head()

In [0]:
# Objetivo:
#
# Validar os requisitos estruturais essenciais.
#
# Justificativa:
#
# Uma divergência de dimensão, nomes duplicados
# ou ausência do target indicaria quebra no contrato
# entre engenharia de dados e ciência de dados.
#
# Ação:
#
# Executa testes explícitos e interrompe o notebook
# se algum critério crítico não for atendido.

EXPECTED_ROWS = 1_966_605
EXPECTED_COLUMNS = 41
TARGET = "in_alfabetizado"

resultados_validacao = {
    "Registros esperados": EXPECTED_ROWS,
    "Registros observados": len(df),
    "Colunas esperadas": EXPECTED_COLUMNS,
    "Colunas observadas": df.shape[1],
    "Nomes únicos": df.columns.is_unique,
    "Target presente": TARGET in df.columns,
    "Atlas presente": any(
        coluna.startswith("atlas_")
        for coluna in df.columns
    ),
    "Censo presente": any(
        coluna.startswith("censo_")
        for coluna in df.columns
    ),
    "Fundeb presente": any(
        coluna.startswith("fundeb_")
        for coluna in df.columns
    ),
    "Indicadores presentes": any(
        coluna.startswith("indicador_")
        for coluna in df.columns
    )
}

for criterio, resultado in resultados_validacao.items():
    print(f"{criterio}: {resultado}")

assert len(df) == EXPECTED_ROWS, (
    "Quantidade de registros divergentes."
)
assert df.shape[1] == EXPECTED_COLUMNS,(
    "Quantidade de colunas divergentes."
)
assert df.columns.is_unique, (
    "Existem nomes de colunas duplicados."
)
assert TARGET in df.columns, (
    "Variável-alvo não encontrada."
)

print("\nValidação estrutural concluída com sucesso.")

In [0]:
# Objetivo:
#
# Confirmar o domínio e a completude do target.
#
# Justificativa:
#
# O problema supervisionado pressupõe um target
# binário, sem categorias inesperadas, composto apenas pelos valores 0 e 1.
#
# Antes da modelagem, também é necessário confirmar
# que não existem valores ausentes ou categorias
# inesperadas na variável-alvo.
#
# Ação:
#
# Identifica o tipo, os valores distintos e a
# quantidade de ausências do target.

valores_target = sorted(
    df[TARGET]
    .dropna()
    .unique()
    .tolist()
)

tipo_target = str(df[TARGET].dtype)

quantidade_valores_distintos = int(
    df[TARGET].nunique(
        dropna=False
    )
)

quantidade_ausentes_target = int(
    df[TARGET].isna().sum()
)

print("VALIDAÇÃO DA VARIÁVEL-ALVO")
print("=" * 50)

print(
    f"Tipo de dado: {tipo_target}"
)

print(
    "Quantidade de valores distintos: "
    f"{quantidade_valores_distintos}"
)

print(
    f"Domínio observado: {valores_target}"
)

print(
    "Valores ausentes: "
    f"{quantidade_ausentes_target:,}"
)

assert set(valores_target).issubset({0,1}), (
    "O target contém valores diferentes de 0 e 1."
)

assert quantidade_ausentes_target == 0, (
    "O target contém valores ausentes"
)

print("\nValidação do target concluída com sucesso.")

## 3. Dicionário e classificação das variáveis

As variáveis serão classificadas por:

1. origem;
2. papel analítico;
3. tipo observado;
4. cardinalidade;
5. elegibilidade preliminar para modelagem.

A classificação inicial não remove nenhuma coluna. Ela organiza a auditoria e será refinada na seção explícita de *data leakage*.

In [0]:
# Objetivo:
#
# Classificar as 41 variáveis por origem e papel.
#
# Justificativa:
#
# Identificadores, variáveis contextuais, target e
# campos ligados à prova exigem tratamentos distintos.
#
# Ação:
#
# Cria um dicionário auditável com tipo, cardinalidade,
# origem e papel preliminar.

identificadores = {
    "id_aluno", "id_escola", "co_municipio",
    "no_municipio", "co_uf", "sg_uf"
}

variaveis_prova_resultado = {
    "in_presenca_lp", "in_preenchimento_lp",
    "co_caderno_lp", "co_bloco_1", "co_bloco_2",
    "co_bloco_3", "co_bloco_4",
    "tx_resposta_bloco_1", "tx_resposta_bloco_2",
    "tx_resposta_bloco_3", "tx_resposta_bloco_4",
    "tx_gabarito_bloco_1", "tx_gabarito_bloco_2",
    "tx_gabarito_bloco_3", "tx_gabarito_bloco_4",
    "vl_peso_aluno_lp", "vl_proficiencia_lp"
}

def identificar_origem(coluna):
    if coluna.startswith("atlas_"):
        return "Atlas"
    if coluna.startswith("censo_"):
        return "Censo Escolar"
    if coluna.startswith("fundeb_"):
        return "Fundeb"
    if coluna.startswith("indicador_"):
        return "Gold Indicadores"
    return "Base de alunos"

def identificar_papel(coluna):
    if coluna == TARGET:
        return "target"
    if coluna in identificadores:
        return "identificador/geográfica"
    if coluna in variaveis_prova_resultado:
        return "prova/resultado - auditar leakage"
    if coluna in {"tp_serie", "tp_dependencia"}:
        return "categórica educacional"
    if coluna == "nu_ano_avaliacao":
        return "temporal"
    if coluna.startswith(("atlas_", "censo_", "fundeb_", "indicador_")):
        return "feature contextual"
    return "feature candidata"

dicionario_variaveis = pd.DataFrame({
    "variavel": df.columns,
    "origem": [identificar_origem(c) for c in df.columns],
    "papel_preliminar": [identificar_papel(c) for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns],
    "cardinalidade": [df[c].nunique(dropna=True) for c in df.columns],
    "ausentes": [df[c].isna().sum() for c in df.columns]
})

dicionario_variaveis["percentual_ausentes"] = (
    100 * dicionario_variaveis["ausentes"] / len(df)
)

display(dicionario_variaveis)

In [0]:
# Objetivo:
#
# Identificar constantes, quase constantes e colunas
# de alta cardinalidade.
#
# Justificativa:
#
# Variáveis constantes não adicionam sinal; alta
# cardinalidade pode exigir exclusão ou encoding
# específico e aumentar o risco de sobreajuste.
#
# Ação:
#
# Calcula indicadores de cardinalidade e concentração.

perfil_cardinalidade = []

for coluna in df.columns:
    frequencias = df[coluna].value_counts(
        normalize=True,
        dropna=False
    )
    perfil_cardinalidade.append({
        "variavel": coluna,
        "cardinalidade": df[coluna].nunique(dropna=True),
        "proporcao_categoria_mais_frequente": (
            frequencias.iloc[0] if len(frequencias) else np.nan
        ),
        "constante": df[coluna].nunique(dropna=False) <= 1,
        "quase_constante_99pct": (
            frequencias.iloc[0] >= 0.99 if len(frequencias) else False
        )
    })

perfil_cardinalidade = pd.DataFrame(
    perfil_cardinalidade
).sort_values(
    ["constante", "quase_constante_99pct", "cardinalidade"],
    ascending=[False, False, False]
)

display(perfil_cardinalidade)

## 4. Qualidade dos dados e valores ausentes

A missingness será analisada em três níveis:

- geral, por variável;
- por origem do enriquecimento;
- por grupos territoriais e educacionais.

Ausências conhecidas decorrentes de cobertura ou chave serão preservadas. Nenhuma imputação será realizada na EDA.

In [0]:
# Objetivo:
#
# Quantificar e ordenar os valores ausentes.
#
# Justificativa:
#
# O percentual de missingness orientará as estratégias
# de imputação e o uso de indicadores de ausência.
#
# Ação:
#
# Produz uma tabela geral de completude.

missing = pd.DataFrame({
    "variavel": df.columns,
    "ausentes": df.isna().sum().values
})

missing["percentual_ausentes"] = (
    100 * missing["ausentes"] / len(df)
)
missing["origem"] = missing["variavel"].map(identificar_origem)
missing = missing.sort_values(
    ["percentual_ausentes", "variavel"],
    ascending=[False, True]
).reset_index(drop=True)

display(missing)

In [0]:
# Objetivo:
#
# Visualizar apenas variáveis com missingness.
#
# Justificativa:
#
# Um gráfico contendo todas as colunas completas
# adicionaria ruído e reduziria a legibilidade.
#
# Ação:
#
# Constrói um ranking percentual das ausências.

missing_plot = missing[missing["ausentes"] > 0].copy()

if not missing_plot.empty:
    plt.figure(figsize=(11, max(4, 0.35 * len(missing_plot))))
    sns.barplot(
        data=missing_plot,
        y="variavel",
        x="percentual_ausentes",
        hue="origem",
        dodge=False
    )
    plt.title("Pergunta: onde estão concentrados os valores ausentes?")
    plt.xlabel("Valores ausentes (%)")
    plt.ylabel("")
    plt.legend(title="Origem", bbox_to_anchor=(1.02, 1))
    plt.tight_layout()
    plt.show()
else:
    print("Não foram encontrados valores ausentes.")

In [0]:
# Objetivo:
#
# Investigar padrões de ausência por UF.
#
# Justificativa:
#
# Ausências concentradas territorialmente podem
# representar cobertura estrutural das fontes,
# e não falha aleatória de qualidade.
#
# Ação:
#
# Calcula o percentual de ausência dos blocos
# enriquecidos por unidade federativa.

features_enriquecidas = [
    c for c in df.columns
    if c.startswith(("atlas_", "censo_", "fundeb_", "indicador_"))
]

missing_por_uf = (
    df.groupby("sg_uf", dropna=False)[features_enriquecidas]
      .agg(lambda serie: 100 * serie.isna().mean())
      .round(3)
)

display(missing_por_uf)

In [0]:
# Objetivo:
#
# Verificar se as ausências dos enriquecimentos
# ocorrem em blocos coerentes com cada fonte.
#
# Justificativa:
#
# As integrações documentaram ausências simultâneas
# por falta de cobertura municipal. A coocorrência
# ajuda a distinguir ausência estrutural de falhas
# isoladas em variáveis específicas.
#
# Ação:
#
# Cria indicadores de bloco ausente e resume suas
# combinações mais frequentes.

blocos = {
    "atlas_ausente": [c for c in df if c.startswith("atlas_")],
    "censo_ausente": [c for c in df if c.startswith("censo_")],
    "fundeb_ausente": [c for c in df if c.startswith("fundeb_")],
    "indicador_ausente": [c for c in df if c.startswith("indicador_")]
}

padrao_ausencias = pd.DataFrame(index=df.index)

for nome, colunas in blocos.items():
    padrao_ausencias[nome] = df[colunas].isna().all(axis=1)

combinacoes_ausencia = (
    padrao_ausencias.value_counts()
    .rename("registros")
    .reset_index()
)
combinacoes_ausencia["percentual"] = (
    100 * combinacoes_ausencia["registros"] / len(df)
)

display(combinacoes_ausencia.head(15))

### Interpretação esperada da missingness

A construção da base já documentou ausências estruturais:

- 510 alunos sem código municipal;
- 1.330 registros sem Atlas;
- 620 sem Censo;
- 661 sem Fundeb;
- 85.271 sem os dois indicadores da Gold.

A EDA deve confirmar esses números. Divergências relevantes devem ser investigadas antes da modelagem. A imputação será aplicada posteriormente dentro do pipeline e ajustada somente sobre o conjunto de treino.

## 5. Análise da variável-alvo

A distribuição de `in_alfabetizado` define a natureza da classificação e orienta:

- métricas de avaliação;
- necessidade de ponderação de classes;
- estratégia de amostragem;
- comparação por grupos.

Além das contagens, serão utilizadas taxas para não confundir desempenho com tamanho populacional.

In [0]:
# Objetivo:
#
# Caracterizar a distribuição absoluta e percentual
# da variável-alvo.
#
# Justificativa:
#
# A proporção das classes orienta métricas, baseline
# e eventual tratamento de desbalanceamento.
#
# Ação:
#
# Calcula contagens, percentuais e baseline majoritário.

distribuicao_target = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="registros")
)

distribuicao_target["percentual"] = (
    100 * distribuicao_target["registros"] / len(df)
)

classe_majoritaria = df[TARGET].value_counts(normalize=True).max()

display(distribuicao_target)
print(f"Baseline da classe majoritária: {classe_majoritaria:.2%}")

In [0]:
# Objetivo:
#
# Visualizar contagem e percentual do target.
#
# Justificativa:
#
# A leitura simultânea evita interpretar apenas
# volume absoluto sem observar o equilíbrio relativo.
#
# Ação:
#
# Constrói dois painéis complementares.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(
    data=distribuicao_target,
    x=TARGET,
    y="registros",
    palette=CORES_TARGET,
    ax=axes[0]
)
axes[0].set_title("Distribuição absoluta do target")
axes[0].set_xlabel("0 = não alfabetizado | 1 = alfabetizado")
axes[0].set_ylabel("Alunos")

sns.barplot(
    data=distribuicao_target,
    x=TARGET,
    y="percentual",
    palette=CORES_TARGET,
    ax=axes[1]
)
axes[1].set_title("Distribuição percentual do target")
axes[1].set_xlabel("0 = não alfabetizado | 1 = alfabetizado")
axes[1].set_ylabel("Percentual (%)")

for ax in axes:
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", padding=3)

plt.tight_layout()
plt.show()

In [0]:
# Objetivo:
#
# Comparar o target por dimensões educacionais.
#
# Justificativa:
#
# Taxas por grupo revelam heterogeneidade sem o
# viés causado por grupos com tamanhos diferentes.
#
# Ação:
#
# Calcula volume e taxa de alfabetização por
# dependência administrativa e série.

def taxa_por_grupo(dataframe, coluna):
    return (
        dataframe.groupby(coluna, dropna=False)[TARGET]
        .agg(alunos="size", alfabetizados="sum", taxa_alfabetizacao="mean")
        .assign(taxa_alfabetizacao_pct=lambda x: 100 * x["taxa_alfabetizacao"])
        .sort_values("taxa_alfabetizacao_pct", ascending=False)
        .reset_index()
    )

taxa_dependencia = taxa_por_grupo(df, "tp_dependencia")
taxa_serie = taxa_por_grupo(df, "tp_serie")

display(taxa_dependencia)
display(taxa_serie)

## 6. Análise univariada

A análise univariada examina distribuição, dispersão, assimetria, extremos, cardinalidade e consistência.

Com quase dois milhões de registros, testes formais de normalidade tenderiam a rejeitar pequenas diferenças sem relevância prática. Por isso, a análise prioriza estatísticas robustas, quantis, assimetria e inspeção gráfica.

In [0]:
# Objetivo:
#
# Produzir estatísticas descritivas das features
# numéricas candidatas.
#
# Justificativa:
#
# Quantis, assimetria e dispersão orientam possíveis
# transformações e detecção de valores extremos.
#
# Ação:
#
# Exclui identificadores numéricos e resume as
# variáveis quantitativas de interesse.

features_contextuais = [
    c for c in df.columns
    if c.startswith(("atlas_", "censo_", "fundeb_", "indicador_"))
]

numericas_contextuais = [
    c for c in features_contextuais
    if pd.api.types.is_numeric_dtype(df[c])
]

resumo_numerico = (
    df[numericas_contextuais]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T
)

resumo_numerico["assimetria"] = df[numericas_contextuais].skew()
resumo_numerico["iqr"] = (
    resumo_numerico["75%"] - resumo_numerico["25%"]
)

display(resumo_numerico)

In [0]:
# Objetivo:
#
# Verificar faixas plausíveis das proporções.
#
# Justificativa:
#
# Indicadores proporcionais devem permanecer entre
# 0 e 1, salvo documentação explícita em contrário.
#
# Ação:
#
# Conta valores abaixo de zero e acima de um.

features_proporcao_0_1 = [
    "censo_prop_mat_2ano_internet_aprendizagem",
    "censo_prop_mat_2ano_alimentacao",
    "censo_prop_mat_2ano_biblioteca_sala_leitura"
]

auditoria_faixas = pd.DataFrame({
    "variavel": features_proporcao_0_1,
    "abaixo_zero": [
        (df[c].dropna() < 0).sum() for c in features_proporcao_0_1
    ],
    "acima_um": [
        (df[c].dropna() > 1).sum() for c in features_proporcao_0_1
    ],
    "minimo": [df[c].min() for c in features_proporcao_0_1],
    "maximo": [df[c].max() for c in features_proporcao_0_1]
})

display(auditoria_faixas)

In [0]:
# Objetivo:
#
# Criar uma amostra exclusiva para visualizações.
#
# Justificativa:
#
# Histogramas e boxplots não precisam renderizar
# quase dois milhões de pontos para representar a
# forma das distribuições.
#
# Ação:
#
# Seleciona amostra reprodutível sem alterar os
# cálculos realizados sobre a base completa.

n_amostra = min(TAMANHO_AMOSTRA_GRAFICOS, len(df))

df_plot = df.sample(
    n=n_amostra,
    random_state=RANDOM_STATE
).copy()

print(f"Amostra visual: {len(df_plot):,} registros")
print("Estatísticas principais continuam calculadas na base completa.")

In [0]:
# Objetivo:
#
# Visualizar distribuições das features contextuais.
#
# Justificativa:
#
# A forma da distribuição indica assimetria,
# concentração, caudas e possível necessidade de
# transformação no pipeline.
#
# Ação:
#
# Produz histogramas organizados por origem.

n_cols = 3
n_rows = int(np.ceil(len(numericas_contextuais) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 3.6 * n_rows)
)
axes = np.array(axes).reshape(-1)

for ax, coluna in zip(axes, numericas_contextuais):
    sns.histplot(
        data=df_plot,
        x=coluna,
        bins=40,
        kde=False,
        ax=ax,
        color="#4C72B0"
    )
    ax.set_title(coluna)
    ax.set_xlabel("")

for ax in axes[len(numericas_contextuais):]:
    ax.set_visible(False)

fig.suptitle(
    "Pergunta: como se distribuem as features contextuais?",
    fontsize=15,
    y=1.01
)
plt.tight_layout()
plt.show()

In [0]:
# Objetivo:
#
# Auditar variáveis categóricas e categorias raras.
#
# Justificativa:
#
# Cardinalidade, inconsistências e categorias pouco
# representadas afetam o encoding e a validação.
#
# Ação:
#
# Resume as principais categorias educacionais e
# territoriais sem usar identificadores individuais.

categoricas_eda = [
    "sg_uf", "tp_serie", "tp_dependencia"
]

resumos_categoricos = []

for coluna in categoricas_eda:
    tabela = (
        df[coluna]
        .value_counts(dropna=False)
        .rename("registros")
        .reset_index()
        .rename(columns={"index": "categoria"})
    )
    tabela["variavel"] = coluna
    tabela["percentual"] = 100 * tabela["registros"] / len(df)
    tabela["categoria_rara_menor_0_1pct"] = tabela["percentual"] < 0.1
    resumos_categoricos.append(tabela)

perfil_categorico = pd.concat(
    resumos_categoricos,
    ignore_index=True
)

display(perfil_categorico)

## 7. Análise bivariada em relação ao target

A relação entre features e target será examinada por:

- diferenças de média e mediana;
- tamanho de efeito padronizado;
- correlação de Spearman;
- taxas de alfabetização por quintis;
- taxas por categorias educacionais.

Como a base é muito grande, valores-p tenderiam a indicar significância para efeitos triviais. A interpretação prioriza magnitude, estabilidade e plausibilidade, não apenas significância estatística.

In [0]:
# Objetivo:
#
# Medir associação exploratória entre features
# numéricas e o target binário.
#
# Justificativa:
#
# Diferenças absolutas, efeito padronizado e
# Spearman oferecem perspectivas complementares.
#
# Ação:
#
# Calcula estatísticas por classe e ordena por
# magnitude do efeito.

def resumo_numerico_target(dataframe, colunas, target):
    linhas = []

    for coluna in colunas:
        base = dataframe[[coluna, target]].dropna()
        grupo_0 = base.loc[base[target] == 0, coluna]
        grupo_1 = base.loc[base[target] == 1, coluna]

        variancia_combinada = (
            (grupo_0.var(ddof=1) + grupo_1.var(ddof=1)) / 2
        )
        desvio_combinado = np.sqrt(variancia_combinada)

        efeito_d = (
            (grupo_1.mean() - grupo_0.mean()) / desvio_combinado
            if desvio_combinado > 0 else np.nan
        )

        correlacao = base[coluna].corr(
            base[target],
            method="spearman"
        )

        linhas.append({
            "variavel": coluna,
            "n_validos": len(base),
            "media_nao_alfabetizado": grupo_0.mean(),
            "media_alfabetizado": grupo_1.mean(),
            "mediana_nao_alfabetizado": grupo_0.median(),
            "mediana_alfabetizado": grupo_1.median(),
            "diferenca_medias": grupo_1.mean() - grupo_0.mean(),
            "efeito_padronizado_d": efeito_d,
            "spearman_target": correlacao
        })

    resultado = pd.DataFrame(linhas)
    resultado["abs_efeito_d"] = resultado["efeito_padronizado_d"].abs()
    return resultado.sort_values("abs_efeito_d", ascending=False)

associacao_numerica_target = resumo_numerico_target(
    df,
    numericas_contextuais,
    TARGET
)

display(associacao_numerica_target)

In [0]:
# Objetivo:
#
# Visualizar diferenças de distribuição por target.
#
# Justificativa:
#
# Boxplots revelam dispersão, sobreposição e extremos,
# complementando as médias agregadas.
#
# Ação:
#
# Seleciona as features de maior efeito exploratório
# e compara as duas classes na amostra gráfica.

top_features_efeito = (
    associacao_numerica_target["variavel"]
    .head(min(8, len(associacao_numerica_target)))
    .tolist()
)

n_cols = 2
n_rows = int(np.ceil(len(top_features_efeito) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 4 * n_rows)
)
axes = np.array(axes).reshape(-1)

for ax, coluna in zip(axes, top_features_efeito):
    sns.boxplot(
        data=df_plot,
        x=TARGET,
        y=coluna,
        palette=CORES_TARGET,
        showfliers=False,
        ax=ax
    )
    ax.set_title(coluna)
    ax.set_xlabel("0 = não alfabetizado | 1 = alfabetizado")

for ax in axes[len(top_features_efeito):]:
    ax.set_visible(False)

fig.suptitle(
    "Pergunta: quais features diferenciam as classes?",
    fontsize=15,
    y=1.01
)
plt.tight_layout()
plt.show()

In [0]:
# Objetivo:
#
# Verificar gradientes de alfabetização ao longo
# das distribuições das features contextuais.
#
# Justificativa:
#
# Quintis permitem identificar relações monotônicas
# ou não lineares sem impor uma forma funcional.
#
# Ação:
#
# Calcula taxa de alfabetização por quintil para
# cada feature numérica com variabilidade suficiente.

taxas_quintis = []

for coluna in numericas_contextuais:
    base = df[[coluna, TARGET]].dropna().copy()

    try:
        base["quintil"] = pd.qcut(
            base[coluna],
            q=5,
            duplicates="drop"
        )
    except ValueError:
        continue

    resumo = (
        base.groupby("quintil", observed=True)[TARGET]
        .agg(alunos="size", taxa_alfabetizacao="mean")
        .reset_index()
    )
    resumo["variavel"] = coluna
    resumo["ordem_quintil"] = np.arange(1, len(resumo) + 1)
    resumo["taxa_alfabetizacao_pct"] = (
        100 * resumo["taxa_alfabetizacao"]
    )
    taxas_quintis.append(resumo)

taxas_quintis = pd.concat(
    taxas_quintis,
    ignore_index=True
)

display(taxas_quintis)

In [0]:
# Objetivo:
#
# Visualizar os gradientes das principais features.
#
# Justificativa:
#
# A direção e regularidade do gradiente ajudam a
# avaliar potencial preditivo e não linearidade.
#
# Ação:
#
# Plota a taxa de alfabetização em cada quintil.

features_quintis_plot = top_features_efeito[:6]

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=taxas_quintis[
        taxas_quintis["variavel"].isin(features_quintis_plot)
    ],
    x="ordem_quintil",
    y="taxa_alfabetizacao_pct",
    hue="variavel",
    marker="o"
)

plt.title(
    "Pergunta: a taxa de alfabetização muda de forma "
    "consistente ao longo das features?"
)
plt.xlabel("Quintil da feature (1 = menor, 5 = maior)")
plt.ylabel("Taxa de alfabetização (%)")
plt.xticks([1, 2, 3, 4, 5])
plt.legend(title="Feature", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [0]:
# Objetivo:
#
# Visualizar a taxa por dependência administrativa.
#
# Justificativa:
#
# A dimensão administrativa pode representar
# diferenças relevantes de contexto educacional.
#
# Ação:
#
# Exibe taxa e volume de cada grupo.

fig, ax1 = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=taxa_dependencia,
    x="tp_dependencia",
    y="taxa_alfabetizacao_pct",
    color="#4C72B0",
    ax=ax1
)

ax1.set_title(
    "Pergunta: a alfabetização varia por dependência administrativa?"
)
ax1.set_xlabel("Tipo de dependência")
ax1.set_ylabel("Taxa de alfabetização (%)")

for container in ax1.containers:
    ax1.bar_label(container, fmt="%.1f", padding=3)

plt.tight_layout()
plt.show()

## 8. Correlações e redundâncias

As correlações serão avaliadas em dois níveis:

1. **nível do aluno:** representa a relação observada na população, mas dá maior peso aos municípios com mais alunos;
2. **nível municipal:** utiliza uma linha por município e evita que a repetição das features contextuais infle artificialmente o peso territorial.

Será utilizada correlação de Spearman, mais robusta a assimetria e relações monotônicas não lineares.

In [0]:
# Objetivo:
#
# Construir a matriz de correlação das features
# contextuais em nível de aluno.
#
# Justificativa:
#
# A matriz identifica agrupamentos e redundâncias
# potenciais, mas ainda reflete o peso populacional.
#
# Ação:
#
# Calcula Spearman na base completa.

corr_aluno = df[numericas_contextuais].corr(method="spearman")

plt.figure(figsize=(13, 10))
sns.heatmap(
    corr_aluno,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=False
)
plt.title("Correlação de Spearman — nível do aluno")
plt.tight_layout()
plt.show()

In [0]:
# Objetivo:
#
# Recalcular correlações em granularidade municipal.
#
# Justificativa:
#
# Features municipais se repetem para muitos alunos.
# Uma linha por município reduz o peso desproporcional
# de localidades com maior população avaliada.
#
# Ação:
#
# Consolida as features por município e calcula
# Spearman no nível territorial.

municipios_contexto = (
    df.dropna(subset=["co_municipio"])
      .groupby(["co_municipio", "sg_uf"], as_index=False)
      .agg({
          **{c: "median" for c in numericas_contextuais},
          TARGET: "mean",
          "id_aluno": "size"
      })
      .rename(columns={
          TARGET: "taxa_alfabetizacao",
          "id_aluno": "alunos"
      })
)

corr_municipio = municipios_contexto[
    numericas_contextuais + ["taxa_alfabetizacao"]
].corr(method="spearman")

plt.figure(figsize=(14, 11))
sns.heatmap(
    corr_municipio,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f"
)
plt.title("Correlação de Spearman — uma linha por município")
plt.tight_layout()
plt.show()

print(f"Municípios analisados: {len(municipios_contexto):,}")

In [0]:
# Objetivo:
#
# Listar pares de features com alta correlação.
#
# Justificativa:
#
# Redundância pode aumentar instabilidade em modelos
# lineares e dificultar a interpretação.
#
# Ação:
#
# Extrai pares com correlação absoluta maior ou
# igual a 0,80 nos níveis aluno e municipal.

def pares_correlacionados(matriz, limite=0.80):
    mascara_superior = np.triu(
        np.ones(matriz.shape, dtype=bool),
        k=1
    )

    pares = (
        matriz.where(mascara_superior)
        .stack()
        .rename("correlacao")
        .reset_index()
        .rename(columns={
            "level_0": "variavel_1",
            "level_1": "variavel_2"
        })
    )

    pares["correlacao_absoluta"] = pares["correlacao"].abs()
    return pares[
        pares["correlacao_absoluta"] >= limite
    ].sort_values("correlacao_absoluta", ascending=False)

redundancias_aluno = pares_correlacionados(corr_aluno)
redundancias_municipio = pares_correlacionados(
    corr_municipio.loc[
        numericas_contextuais,
        numericas_contextuais
    ]
)

print("Pares altamente correlacionados no nível do aluno:")
display(redundancias_aluno)

print("Pares altamente correlacionados no nível municipal:")
display(redundancias_municipio)

## 9. Análise territorial

A análise territorial deve identificar heterogeneidade e vulnerabilidade sem criar rankings instáveis.

Para municípios, será aplicado um volume mínimo de alunos. O ranking não representa causalidade nem qualidade isolada da gestão municipal; ele é um instrumento exploratório para priorização e investigação.

In [0]:
# Objetivo:
#
# Comparar taxas de alfabetização entre UFs.
#
# Justificativa:
#
# As taxas permitem avaliar heterogeneidade
# territorial controlando o tamanho dos grupos.
#
# Ação:
#
# Calcula volume, alfabetizados e taxa por UF.

taxa_uf = (
    df.groupby("sg_uf", dropna=False)[TARGET]
      .agg(alunos="size", alfabetizados="sum", taxa_alfabetizacao="mean")
      .assign(
          taxa_alfabetizacao_pct=lambda x: 100 * x["taxa_alfabetizacao"]
      )
      .sort_values("taxa_alfabetizacao_pct", ascending=False)
      .reset_index()
)

display(taxa_uf)

In [0]:
# Objetivo:
#
# Visualizar o ranking de taxa por UF.
#
# Justificativa:
#
# O ranking evidencia amplitude territorial e
# unidades que merecem investigação contextual.
#
# Ação:
#
# Ordena as UFs e inclui a média nacional como
# referência visual.

media_nacional = 100 * df[TARGET].mean()

plt.figure(figsize=(11, 9))
sns.barplot(
    data=taxa_uf,
    y="sg_uf",
    x="taxa_alfabetizacao_pct",
    color="#4C72B0"
)
plt.axvline(
    media_nacional,
    color="#C44E52",
    linestyle="--",
    label=f"Média nacional: {media_nacional:.1f}%"
)
plt.title("Pergunta: como a alfabetização varia entre as UFs?")
plt.xlabel("Taxa de alfabetização (%)")
plt.ylabel("")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# Objetivo:
#
# Identificar municípios com taxas extremas usando
# um limite mínimo de observações.
#
# Justificativa:
#
# Rankings sem filtro podem destacar municípios com
# poucos alunos e alta variabilidade amostral.
#
# Ação:
#
# Calcula taxas municipais e apresenta os extremos
# entre municípios com pelo menos 100 alunos.

taxa_municipio = (
    df.dropna(subset=["co_municipio"])
      .groupby(
          ["co_municipio", "no_municipio", "sg_uf"],
          as_index=False
      )[TARGET]
      .agg(alunos="size", alfabetizados="sum", taxa_alfabetizacao="mean")
)

taxa_municipio["taxa_alfabetizacao_pct"] = (
    100 * taxa_municipio["taxa_alfabetizacao"]
)

municipios_elegiveis = taxa_municipio[
    taxa_municipio["alunos"] >= MIN_ALUNOS_MUNICIPIO
].copy()

print("Municípios elegíveis:", len(municipios_elegiveis))
print("Maiores taxas:")
display(
    municipios_elegiveis.nlargest(
        15,
        "taxa_alfabetizacao_pct"
    )
)

print("Menores taxas:")
display(
    municipios_elegiveis.nsmallest(
        15,
        "taxa_alfabetizacao_pct"
    )
)

In [0]:
# Objetivo:
#
# Criar uma visão exploratória de vulnerabilidade.
#
# Justificativa:
#
# A combinação entre desempenho atual e contexto
# socioeconômico ajuda a localizar grupos prioritários.
#
# Ação:
#
# Compara taxa municipal com IDHM e vulnerabilidade
# infantil, mantendo o tamanho do município no gráfico.

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=municipios_contexto[
        municipios_contexto["alunos"] >= MIN_ALUNOS_MUNICIPIO
    ],
    x="atlas_idhm",
    y="taxa_alfabetizacao",
    hue="atlas_prop_pobreza_criancas",
    size="alunos",
    sizes=(20, 250),
    alpha=0.65,
    palette="viridis_r"
)

plt.title(
    "Pergunta: desempenho e vulnerabilidade "
    "socioeconômica apresentam padrões territoriais?"
)
plt.xlabel("IDHM municipal — Atlas 2010")
plt.ylabel("Taxa municipal de alfabetização em 2025")
plt.legend(bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 10. Análise por origem do enriquecimento

Esta seção consolida a contribuição exploratória de cada fonte:

- **Atlas:** desenvolvimento humano, renda, desigualdade e vulnerabilidade;
- **Censo:** infraestrutura/condições educacionais;
- **Fundeb:** escala financeira;
- **Indicadores:** contexto municipal de alfabetização em 2024 e meta de 2025.

As features do Atlas são temporalmente defasadas (2010), enquanto Censo, Fundeb e o indicador de alfabetização representam principalmente 2024.

In [0]:
# Objetivo:
#
# Resumir a associação das features por origem.
#
# Justificativa:
#
# A comparação por bloco permite avaliar se cada
# enriquecimento adiciona sinal exploratório.
#
# Ação:
#
# Agrega magnitude média e máxima dos efeitos e
# correlações com o target.

resumo_por_origem = (
    associacao_numerica_target
    .assign(
        origem=lambda x: x["variavel"].map(identificar_origem),
        abs_spearman=lambda x: x["spearman_target"].abs()
    )
    .groupby("origem")
    .agg(
        quantidade_features=("variavel", "size"),
        efeito_d_medio=("abs_efeito_d", "mean"),
        efeito_d_maximo=("abs_efeito_d", "max"),
        spearman_medio=("abs_spearman", "mean"),
        spearman_maximo=("abs_spearman", "max")
    )
    .sort_values("efeito_d_maximo", ascending=False)
    .reset_index()
)

display(resumo_por_origem)

In [0]:
# Objetivo:
#
# Avaliar a assimetria das variáveis financeiras.
#
# Justificativa:
#
# Receitas municipais costumam apresentar cauda
# longa e podem se beneficiar de log1p na modelagem.
#
# Ação:
#
# Compara assimetria antes e depois da transformação
# sem modificar a base original.

features_fundeb = [
    c for c in df.columns if c.startswith("fundeb_")
]

auditoria_log_fundeb = []

for coluna in features_fundeb:
    serie = df[coluna].dropna()
    serie_nao_negativa = serie.clip(lower=0)

    auditoria_log_fundeb.append({
        "variavel": coluna,
        "minimo": serie.min(),
        "maximo": serie.max(),
        "assimetria_original": serie.skew(),
        "assimetria_log1p": np.log1p(serie_nao_negativa).skew(),
        "valores_negativos": int((serie < 0).sum())
    })

auditoria_log_fundeb = pd.DataFrame(auditoria_log_fundeb)
display(auditoria_log_fundeb)

In [0]:
# Objetivo:
#
# Comparar a taxa de 2025 com o contexto municipal
# de 2024 e a meta municipal de 2025.
#
# Justificativa:
#
# Essas variáveis possuem forte relevância temporal,
# mas devem ser interpretadas no nível municipal.
#
# Ação:
#
# Calcula correlações territoriais e diferença entre
# resultado observado e meta.

indicadores_temporais = [
    "indicador_meta_final_2025",
    "indicador_pc_aluno_alfabetizado_2024"
]

analise_indicadores_municipais = municipios_contexto[
    ["co_municipio", "sg_uf", "alunos", "taxa_alfabetizacao"]
    + indicadores_temporais
].copy()

analise_indicadores_municipais["taxa_alfabetizacao_pct"] = (
    100 * analise_indicadores_municipais["taxa_alfabetizacao"]
)

analise_indicadores_municipais["gap_meta_2025"] = (
    analise_indicadores_municipais["taxa_alfabetizacao_pct"]
    - analise_indicadores_municipais["indicador_meta_final_2025"]
)

display(
    analise_indicadores_municipais[
        ["taxa_alfabetizacao_pct"] + indicadores_temporais + ["gap_meta_2025"]
    ].corr(method="spearman")
)

## 11. Auditoria explícita de data leakage

A previsão deve usar apenas informações que estariam disponíveis no momento real da decisão.

Variáveis produzidas pela aplicação/correção da prova de 2025 podem reconstruir diretamente o target e gerar desempenho artificialmente alto. Nenhuma variável será removida silenciosamente: a tabela abaixo registra a decisão recomendada e sua justificativa.

In [0]:
# Objetivo:
#
# Classificar as variáveis quanto ao uso no modelo.
#
# Justificativa:
#
# Identificadores, target e campos derivados da prova
# podem causar leakage, memorização ou redundância.
#
# Ação:
#
# Constrói uma matriz explícita de decisão.

decisoes_leakage = {
    "in_alfabetizado": (
        "excluir do conjunto X",
        "É a variável-alvo."
    ),
    "id_aluno": (
        "excluir",
        "Identificador individual sem significado preditivo generalizável."
    ),
    "id_escola": (
        "investigar/excluir no baseline",
        "Alta cardinalidade e risco de memorização da escola."
    ),
    "co_municipio": (
        "investigar",
        "Útil para agrupamento do split; encoding direto pode memorizar território."
    ),
    "no_municipio": (
        "excluir",
        "Texto redundante com o código municipal e alta cardinalidade."
    ),
    "co_uf": (
        "excluir por redundância",
        "Redundante com sg_uf."
    ),
    "sg_uf": (
        "manter",
        "Contexto territorial disponível antes da avaliação."
    ),
    "nu_ano_avaliacao": (
        "excluir se constante",
        "Sem variabilidade em uma base contendo apenas 2025."
    ),
    "in_presenca_lp": (
        "excluir",
        "Informação da aplicação da prova; não disponível em predição antecipada."
    ),
    "in_preenchimento_lp": (
        "excluir",
        "Informação da aplicação/preenchimento da prova."
    ),
    "co_caderno_lp": (
        "excluir",
        "Artefato operacional da prova, sem valor causal para intervenção."
    ),
    "vl_peso_aluno_lp": (
        "investigar/excluir no baseline",
        "Peso amostral pode ser usado na análise, mas sua disponibilidade em produção deve ser confirmada."
    ),
    "vl_proficiencia_lp": (
        "excluir",
        "Resultado direto da avaliação usado para definir alfabetização."
    )
}

for bloco in range(1, 5):
    decisoes_leakage[f"co_bloco_{bloco}"] = (
        "excluir",
        "Estrutura do instrumento aplicado na prova."
    )
    decisoes_leakage[f"tx_resposta_bloco_{bloco}"] = (
        "excluir",
        "Resposta do aluno; informação posterior e diretamente ligada ao resultado."
    )
    decisoes_leakage[f"tx_gabarito_bloco_{bloco}"] = (
        "excluir",
        "Gabarito da prova; permite reconstruir acertos e o target."
    )

auditoria_leakage = pd.DataFrame([
    {
        "variavel": coluna,
        "decisao_recomendada": decisao,
        "justificativa": justificativa
    }
    for coluna, (decisao, justificativa) in decisoes_leakage.items()
]).sort_values(["decisao_recomendada", "variavel"])

display(auditoria_leakage)

In [0]:
# Objetivo:
#
# Definir o conjunto preliminar de features seguras.
#
# Justificativa:
#
# A modelagem precisa iniciar com um baseline que
# não dependa de informações posteriores à prova.
#
# Ação:
#
# Separa features candidatas, excluídas e sob
# investigação sem alterar a base original.

excluir_modelo = set(
    auditoria_leakage.loc[
        auditoria_leakage["decisao_recomendada"].str.startswith("excluir"),
        "variavel"
    ]
)

investigar_modelo = set(
    auditoria_leakage.loc[
        auditoria_leakage["decisao_recomendada"].str.startswith("investigar"),
        "variavel"
    ]
)

features_seguras_preliminares = [
    c for c in df.columns
    if c != TARGET
    and c not in excluir_modelo
    and c not in investigar_modelo
]

resumo_elegibilidade = pd.DataFrame({
    "grupo": [
        "Features seguras preliminares",
        "Excluir",
        "Investigar antes de usar"
    ],
    "quantidade": [
        len(features_seguras_preliminares),
        len(excluir_modelo),
        len(investigar_modelo)
    ]
})

display(resumo_elegibilidade)
print("Features seguras preliminares:")
print(features_seguras_preliminares)

## 12. Hipóteses analíticas

As hipóteses abaixo transformam a exploração em perguntas testáveis para a modelagem.

Cada hipótese deve ser avaliada com:

- evidência exploratória;
- limitação;
- forma de teste;
- utilidade potencial.

O resultado da EDA não prova causalidade.

In [0]:
# Objetivo:
#
# Registrar hipóteses testáveis e sua ligação com
# as etapas posteriores.
#
# Justificativa:
#
# Hipóteses explícitas evitam uma modelagem guiada
# apenas por métricas e favorecem interpretabilidade.
#
# Ação:
#
# Consolida perguntas, evidências necessárias,
# limitações e testes futuros.

hipoteses = pd.DataFrame([
    {
        "id": "H1",
        "hipotese": (
            "Maior desenvolvimento humano e educacional municipal "
            "está associado a maior alfabetização."
        ),
        "features": "atlas_idhm; atlas_idhm_e",
        "evidencia_eda": (
            "Gradiente por quintis, efeito padronizado e Spearman municipal."
        ),
        "limitacao": "Atlas de 2010 e natureza observacional.",
        "teste_modelagem": (
            "Importância/SHAP, desempenho com e sem o bloco Atlas."
        )
    },
    {
        "id": "H2",
        "hipotese": (
            "Maior vulnerabilidade e desigualdade estão associadas "
            "a menor taxa de alfabetização."
        ),
        "features": (
            "atlas_indice_gini; atlas_prop_pobreza_criancas; "
            "atlas_taxa_criancas_dom_sem_fund"
        ),
        "evidencia_eda": "Direção do gradiente e correlação municipal.",
        "limitacao": "Confundimento territorial e defasagem temporal.",
        "teste_modelagem": "SHAP, relações parciais e estabilidade por UF."
    },
    {
        "id": "H3",
        "hipotese": (
            "Melhores condições educacionais municipais do Censo "
            "estão associadas a maior alfabetização."
        ),
        "features": "features censo_",
        "evidencia_eda": "Taxas por quintis e efeito entre classes.",
        "limitacao": (
            "Indicadores agregados; não identificam a escola do aluno."
        ),
        "teste_modelagem": (
            "Importância do bloco e validação agrupada por município."
        )
    },
    {
        "id": "H4",
        "hipotese": (
            "As receitas do Fundeb agregam contexto financeiro, "
            "mas exigem transformação de escala."
        ),
        "features": "features fundeb_",
        "evidencia_eda": "Assimetria e comparação original versus log1p.",
        "limitacao": (
            "Valores absolutos refletem porte populacional e não gasto por aluno."
        ),
        "teste_modelagem": (
            "Comparar features originais/log1p e avaliar evolução per capita futura."
        )
    },
    {
        "id": "H5",
        "hipotese": (
            "O indicador municipal de alfabetização de 2024 melhora "
            "a predição individual de 2025."
        ),
        "features": "indicador_pc_aluno_alfabetizado_2024",
        "evidencia_eda": (
            "Associação territorial com taxa individual agregada de 2025."
        ),
        "limitacao": (
            "Feature municipal repetida e possível dominância territorial."
        ),
        "teste_modelagem": (
            "Ablation study e validação por município."
        )
    },
    {
        "id": "H6",
        "hipotese": (
            "Existem diferenças relevantes por UF e dependência administrativa."
        ),
        "features": "sg_uf; tp_dependencia",
        "evidencia_eda": "Amplitude das taxas por grupo.",
        "limitacao": "Associação pode refletir composição socioeconômica.",
        "teste_modelagem": (
            "Interações, métricas por subgrupo e análise de equidade."
        )
    }
])

display(hipoteses)

## 13. Recomendações para preparação e modelagem

A EDA deve terminar com decisões concretas. As recomendações abaixo formam o contrato inicial da próxima etapa.

In [0]:
# Objetivo:
#
# Consolidar decisões práticas para a pipeline.
#
# Justificativa:
#
# Os achados exploratórios precisam orientar ações
# reproduzíveis e não tratamentos ad hoc.
#
# Ação:
#
# Cria uma matriz de recomendações para preparação,
# validação, métricas e interpretabilidade.

recomendacoes_modelagem = pd.DataFrame([
    {
        "tema": "Base imutável",
        "decisao": (
            "Não sobrescrever a base enriquecida; criar artefatos derivados."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Target",
        "decisao": (
            "Usar in_alfabetizado apenas como y; verificar classes antes do split."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Leakage",
        "decisao": (
            "Excluir proficiência, respostas, gabaritos, presença, "
            "preenchimento, caderno e blocos do baseline."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Identificadores",
        "decisao": (
            "Excluir id_aluno e nome do município; usar co_municipio "
            "principalmente como grupo de validação."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Split",
        "decisao": (
            "Priorizar StratifiedGroupKFold/GroupShuffleSplit por município; "
            "comparar com split estratificado para alunos de municípios conhecidos."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Missing numérico",
        "decisao": (
            "Imputar dentro do pipeline, ajustando somente no treino; "
            "testar mediana e indicadores de ausência por bloco."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Categóricas",
        "decisao": (
            "Aplicar encoding dentro do ColumnTransformer; tratar categoria ausente."
        ),
        "prioridade": "Alta"
    },
    {
        "tema": "Fundeb",
        "decisao": (
            "Testar log1p se a assimetria for alta; futuramente avaliar "
            "receita por aluno/população."
        ),
        "prioridade": "Alta"
    },
    {
        "tema": "Redundância",
        "decisao": (
            "Não excluir apenas por correlação; avaliar estabilidade, "
            "regularização e importância por permutação/SHAP."
        ),
        "prioridade": "Alta"
    },
    {
        "tema": "Métricas",
        "decisao": (
            "Reportar ROC-AUC, PR-AUC, F1, recall, precisão, matriz de "
            "confusão e métricas por UF/dependência."
        ),
        "prioridade": "Obrigatória"
    },
    {
        "tema": "Baseline",
        "decisao": (
            "Comparar DummyClassifier, Regressão Logística e modelo "
            "baseado em árvores."
        ),
        "prioridade": "Alta"
    },
    {
        "tema": "Interpretabilidade",
        "decisao": (
            "Usar permutation importance e SHAP; separar associação de causalidade."
        ),
        "prioridade": "Alta"
    },
    {
        "tema": "Granularidade",
        "decisao": (
            "Avaliar resultados em nível de aluno e agregados por município."
        ),
        "prioridade": "Obrigatória"
    }
])

display(recomendacoes_modelagem)

In [0]:
# Objetivo:
#
# Persistir as evidências tabulares da EDA.
#
# Justificativa:
#
# Tabelas de diagnóstico e decisão tornam a análise
# auditável e reutilizável na modelagem e no README.
#
# Ação:
#
# Grava apenas artefatos derivados, sem modificar
# a base enriquecida de referência.

artefatos_eda = {
    "dicionario_variaveis.csv": dicionario_variaveis,
    "missing_values.csv": missing,
    "distribuicao_target.csv": distribuicao_target,
    "associacao_numerica_target.csv": associacao_numerica_target,
    "redundancias_nivel_aluno.csv": redundancias_aluno,
    "redundancias_nivel_municipio.csv": redundancias_municipio,
    "taxa_alfabetizacao_uf.csv": taxa_uf,
    "taxa_alfabetizacao_municipio.csv": taxa_municipio,
    "auditoria_data_leakage.csv": auditoria_leakage,
    "hipoteses_analiticas.csv": hipoteses,
    "recomendacoes_modelagem.csv": recomendacoes_modelagem
}

for nome_arquivo, tabela in artefatos_eda.items():
    caminho = f"{EDA_OUTPUT_PATH}/{nome_arquivo}"
    tabela.to_csv(
        caminho,
        sep=";",
        encoding="utf-8-sig",
        index=False
    )
    print("Salvo:", caminho)

## 14. Conclusão analítica

A conclusão deve sintetizar evidências, riscos e decisões. O bloco abaixo produz um resumo dinâmico com base nos resultados efetivamente calculados, evitando registrar números antes da execução da EDA.

In [0]:
# Objetivo:
#
# Gerar uma síntese executiva baseada nos resultados
# observados durante a execução.
#
# Justificativa:
#
# A EDA deve encerrar com decisões e não apenas com
# gráficos ou tabelas isoladas.
#
# Ação:
#
# Consolida perfil do target, missingness, features
# de maior associação, amplitude territorial e riscos.

taxa_target = 100 * df[TARGET].mean()
maior_missing = missing.iloc[0]
top_associacoes = (
    associacao_numerica_target["variavel"]
    .head(5)
    .tolist()
)
uf_maior = taxa_uf.iloc[0]
uf_menor = taxa_uf.iloc[-1]

print("CONCLUSÃO EXECUTIVA DA EDA")
print("=" * 80)
print(f"População analisada: {len(df):,} alunos e {df.shape[1]} variáveis.")
print(f"Taxa geral de alfabetização: {taxa_target:.2f}%.")
print(
    "Maior missingness: "
    f"{maior_missing['variavel']} "
    f"({maior_missing['percentual_ausentes']:.2f}%)."
)
print(
    "Features contextuais com maior efeito exploratório: "
    + ", ".join(top_associacoes)
    + "."
)
print(
    "Amplitude entre UFs: "
    f"{uf_maior['sg_uf']} = {uf_maior['taxa_alfabetizacao_pct']:.2f}% "
    f"e {uf_menor['sg_uf']} = {uf_menor['taxa_alfabetizacao_pct']:.2f}%."
)
print(
    f"Variáveis recomendadas para exclusão: {len(excluir_modelo)}; "
    f"sob investigação: {len(investigar_modelo)}."
)
print(
    "Decisão central: construir o baseline sem variáveis posteriores à prova, "
    "com imputação e encoding dentro do pipeline e validação agrupada por "
    "município para reduzir leakage territorial."
)
print(
    "As relações observadas são associativas e deverão ser confirmadas por "
    "validação fora da amostra, ablation studies e interpretabilidade."
)

### Critério de encerramento

A EDA estará concluída quando os resultados executados permitirem responder, com evidências:

1. como o target se distribui;
2. quais grupos apresentam diferenças relevantes;
3. quais features demonstram associação exploratória;
4. onde estão as ausências e se são estruturais;
5. quais variáveis são redundantes;
6. quais campos apresentam risco de *data leakage*;
7. quais hipóteses seguirão para a modelagem;
8. quais decisões de pré-processamento decorrem da exploração.

O próximo artefato deverá implementar essas decisões em uma pipeline reproduzível, sem sobrescrever a base analítica enriquecida.